# GPR Buried-Object Detection: Phase 2 Benchmarking & Grouped 5-Fold Ablation Study

This notebook runs the complete Phase 2 experimental revisions on a single Kaggle GPU (`device=0`):
1. **Model Comparison**: Benchmarks **YOLOv8n (Proposed)**, **YOLOv5nu (Predecessor)**, **YOLO11n (Next-Gen)**, and **YOLOv8s (Capacity Scaling)** across all 5 folds of the honest grouped cross-validation scheme (`split_5fold.json`).
2. **Gate 2 Verification**: Automatically verifies that the Kaggle GPU YOLOv8n run matches the Laptop CPU baseline within the 0.02 mAP50 threshold.
3. **Leave-One-Out (LOO) & No-Augmentation Ablation**: Re-runs all 7 ablation configurations across the 5 folds with exact suffix mappings.
4. **Packaging**: Generates publication-ready IEEEtran LaTeX tables, comparison CSVs, and archives all outputs into `/kaggle/working/results.zip`.

### Instructions:
1. Under **Session options** (right pane) -> set **Accelerator** to **GPU T4 x2** or **GPU P100**.
2. Turn **Internet** ON in the right sidebar.
3. Dataset uploaded & extracted into `/kaggle/working` (or attached via `/kaggle/input`).
4. Click **Run All**.

In [ ]:
# 1. GPU Environment Verification & Package Setup
!nvidia-smi

import os
import sys
import shutil
from pathlib import Path

# Install ultralytics
!pip install -q ultralytics

import torch
import ultralytics
print(f"PyTorch: {torch.__version__} | Ultralytics: {ultralytics.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU (device 0): {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Locate Dataset / Repository and Set up Working Directory
import os
import shutil
import zipfile
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")

def is_repo_root(p: Path) -> bool:
    """Check if directory contains the project files."""
    return (p / "experiments").is_dir() and ((p / "datasets").is_dir() or (p / "GPR_data").is_dir() or (p / "config.py").is_file())

repo_dir = None

# Priority 1: Check if extracted directly inside /kaggle/working
if is_repo_root(KAGGLE_WORKING):
    print("Found project directly inside /kaggle/working!")
    repo_dir = KAGGLE_WORKING

# Priority 2: Check subdirectories of /kaggle/working
if repo_dir is None and KAGGLE_WORKING.exists():
    for sub in KAGGLE_WORKING.iterdir():
        if sub.is_dir() and is_repo_root(sub):
            print(f"Found project in subdirectory: {sub}")
            repo_dir = sub
            break

# Priority 3: Check if there is an unextracted .zip in /kaggle/working
if repo_dir is None and KAGGLE_WORKING.exists():
    zips = list(KAGGLE_WORKING.glob("*.zip"))
    if zips:
        chosen_zip = zips[0]
        print(f"Found zip archive {chosen_zip.name} in /kaggle/working. Extracting into /kaggle/working/repo...")
        extract_dest = KAGGLE_WORKING / "repo"
        extract_dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(chosen_zip, 'r') as zf:
            zf.extractall(extract_dest)
        if is_repo_root(extract_dest):
            repo_dir = extract_dest
        else:
            for sub in extract_dest.iterdir():
                if sub.is_dir() and is_repo_root(sub):
                    repo_dir = sub
                    break

# Priority 4: Fallback to /kaggle/input (attached Kaggle dataset)
if repo_dir is None and KAGGLE_INPUT.exists():
    for d in KAGGLE_INPUT.iterdir():
        if d.is_dir():
            src = d if is_repo_root(d) else None
            if not src:
                for sub in d.iterdir():
                    if sub.is_dir() and is_repo_root(sub):
                        src = sub
                        break
            if src:
                print(f"Found dataset in /kaggle/input: {src}")
                dest = KAGGLE_WORKING / "repo"
                if dest.exists():
                    shutil.rmtree(dest)
                print(f"Copying to writable working directory: {dest}...")
                shutil.copytree(src, dest, ignore=shutil.ignore_patterns('runs', '*.zip', '.git'))
                repo_dir = dest
                break

if repo_dir is None:
    raise FileNotFoundError("Could not find project directory with 'experiments/' and 'datasets/'. Please verify your extracted files in /kaggle/working.")

os.chdir(repo_dir)
print(f"Active Project Directory: {os.getcwd()}")

# -------------------------------------------------------------------------
# CRITICAL: Auto-fix Windows absolute paths in datasets/split_5fold/*/data.yaml
# -------------------------------------------------------------------------
print("\nVerifying and updating data.yaml paths for 5-fold cross validation...")
fixed_count = 0
for f_idx in range(1, 6):
    yaml_path = Path(f"datasets/split_5fold/fold_{f_idx}/data.yaml")
    if yaml_path.exists():
        abs_fold = yaml_path.parent.resolve().as_posix()
        yaml_content = f"""path: {abs_fold}
train: images/train
val: images/val
test: images/test

names:
  0: utility
  1: cavity
"""
        yaml_path.write_text(yaml_content, encoding="utf-8")
        print(f"  [OK] Fold {f_idx}: set path to {abs_fold}")
        fixed_count += 1
    else:
        print(f"  [WARNING] {yaml_path} not found.")

print(f"Done! Updated {fixed_count} data.yaml configuration(s).")

In [ ]:
# 3. Run Phase 2 Model Benchmarking (YOLOv8n, YOLOv5nu, YOLO11n, YOLOv8s across 5 folds)
# All models trained on the exact same GPU hardware for fair, apples-to-apples comparison.
# Includes Gate 2 verification comparing Kaggle GPU YOLOv8n vs. Laptop CPU baseline.
!python experiments/02_train_all_models.py --epochs 50 --device 0 --models yolov8n yolov5nu yolo11n yolov8s

In [ ]:
# 4. Run Grouped 5-Fold Leave-One-Out (LOO) & No-Augmentation Ablation Study
!python experiments/02_run_ablations.py --epochs 50 --device 0

In [ ]:
# 5. Display Model Comparison & Ablation Tables
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

comp_csv = Path("results/model_comparison/comparison.csv")
if comp_csv.exists():
    print("\n" + "="*60)
    print("MODEL COMPARISON BENCHMARK (GROUPED 5-FOLD)")
    print("="*60)
    display(pd.read_csv(comp_csv))
else:
    print(f"{comp_csv} not found yet.")

abl_csv = Path("results/ablation_study/ablation_summary.csv")
if abl_csv.exists():
    print("\n" + "="*60)
    print("LEAVE-ONE-OUT ABLATION STUDY (GROUPED 5-FOLD)")
    print("="*60)
    display(pd.read_csv(abl_csv))
else:
    print(f"{abl_csv} not found yet.")

In [ ]:
# 6. Archive all results into /kaggle/working/results.zip for Easy Download
import os
import zipfile
from pathlib import Path

results_dir = Path("results").resolve()
zip_dest = Path("/kaggle/working/results.zip")

if not results_dir.exists():
    print(f"[WARNING] {results_dir} does not exist yet. Please check training progress.")
else:
    print(f"Archiving {results_dir} into {zip_dest}...")
    with zipfile.ZipFile(zip_dest, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                abs_path = Path(root) / file
                rel_path = abs_path.relative_to(results_dir.parent)
                zipf.write(abs_path, rel_path)

    size_mb = zip_dest.stat().st_size / (1024 * 1024)
    print(f"Done! Created {zip_dest} ({size_mb:.2f} MB).")
    print("Download results.zip from the Kaggle Output sidebar on the right.")